In [1]:
sc.install_pypi_package("pandas")
sc.install_pypi_package("numpy")
sc.install_pypi_package("boto3")


VBox()

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
5,application_1732062475289_0006,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

  Attempting uninstall: python-dateutil
    Found existing installation: python-dateutil 2.8.1
    Not uninstalling python-dateutil at /usr/lib/python3.9/site-packages, outside environment /mnt/yarn/usercache/livy/appcache/application_1732062475289_0006/container_1732062475289_0006_01_000001/tmp/spark-4742e2d7-ae97-4353-acce-90484512297d
    Can't uninstall 'python-dateutil'. No files were found to uninstall.



ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
awscli 2.15.30 requires python-dateutil<=2.8.2,>=2.1, but you have python-dateutil 2.9.0.post0 which is incompatible.



In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, when, desc, sum
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import KMeans


VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## Initialize Spark Session

In [3]:
spark = SparkSession.builder \
    .appName("Yelp Recommendation System with KMeans") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

print(spark)


VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## Load Dataset

In [4]:
data_path = "s3://yelp-final-raja/processed_yelp_data_single/yelp-pre-processed.csv"

data = spark.read.csv(data_path, header=True, inferSchema=True)

data.printSchema()
data.show(10)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- UserID: integer (nullable = true)
 |-- ItemID: string (nullable = true)
 |-- Rating: double (nullable = true)
 |-- NumberReview: integer (nullable = true)
 |-- State: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Category: string (nullable = true)

+------+--------------------+------+------------+-----+---------+--------+
|UserID|              ItemID|Rating|NumberReview|State|     City|Category|
+------+--------------------+------+------------+-----+---------+--------+
|   124|         Dragon City|   3.5|          39|   AL|   Daphne|Delivery|
|   171|      Janino's Pizza|   3.5|          30|   AL|   Daphne|Delivery|
|   179|Roll & Go Sushi A...|   4.0|          10|   AL|   Daphne|Delivery|
|   187|       Marco's Pizza|   3.0|          25|   FL|Pensacola|Delivery|
|   196|Santino's Pizza &...|   4.0|           5|   FL|   Milton|Delivery|
|   220|                 KFC|   3.0|           4|   AL| Saraland|Delivery|
|   246|   Godfather's Pizza|   4.0|          

## Explore Data

In [5]:
print(f"Total Rows: {data.count()}")

print(f"Unique Users: {data.select('UserID').distinct().count()}")
print(f"Unique Items: {data.select('ItemID').distinct().count()}")

data.describe(["Rating", "NumberReview"]).show()

data.select(
    [sum(col(c).isNull().cast("int")).alias(c) for c in data.columns]
).show()


VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Total Rows: 998942
Unique Users: 998942
Unique Items: 190583
+-------+-----------------+------------------+
|summary|           Rating|      NumberReview|
+-------+-----------------+------------------+
|  count|           998942|            998942|
|   mean|2.786716345893956| 56.74496917738968|
| stddev|1.913290849120266|185.66041957882004|
|    min|              0.0|                 0|
|    max|              5.0|             12570|
+-------+-----------------+------------------+

+------+------+------+------------+-----+----+--------+
|UserID|ItemID|Rating|NumberReview|State|City|Category|
+------+------+------+------------+-----+----+--------+
|     0|     0|     0|           0|    0|   0|       0|
+------+------+------+------------+-----+----+--------+

## Prepare Features for Clustering

In [6]:
assembler = VectorAssembler(
    inputCols=["Rating", "NumberReview"],
    outputCol="features"
)
cluster_data = assembler.transform(data)

cluster_data.select("features").show(10, truncate=False)


VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----------+
|features  |
+----------+
|[3.5,39.0]|
|[3.5,30.0]|
|[4.0,10.0]|
|[3.0,25.0]|
|[4.0,5.0] |
|[3.0,4.0] |
|[4.0,17.0]|
|[2.0,10.0]|
|[3.5,11.0]|
|[2.5,23.0]|
+----------+
only showing top 10 rows

## Train KMeans Model

In [7]:
kmeans = KMeans(k=10, seed=42, featuresCol="features", predictionCol="cluster")
kmeans_model = kmeans.fit(cluster_data)

clustered_data = kmeans_model.transform(cluster_data)

clustered_data.select("ItemID", "City", "Category", "Rating", "cluster").show(10, truncate=False)


VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-----------------------------+---------+--------+------+-------+
|ItemID                       |City     |Category|Rating|cluster|
+-----------------------------+---------+--------+------+-------+
|Dragon City                  |Daphne   |Delivery|3.5   |7      |
|Janino's Pizza               |Daphne   |Delivery|3.5   |1      |
|Roll & Go Sushi Asian Kitchen|Daphne   |Delivery|4.0   |1      |
|Marco's Pizza                |Pensacola|Delivery|3.0   |1      |
|Santino's Pizza & Grinders   |Milton   |Delivery|4.0   |1      |
|KFC                          |Saraland |Delivery|3.0   |1      |
|Godfather's Pizza            |Mobile   |Delivery|4.0   |1      |
|Domino's Pizza               |Daphne   |Delivery|2.0   |1      |
|Hungry Howie's Pizza         |Mobile   |Delivery|3.5   |1      |
|Shang Hai II                 |Pensacola|Delivery|2.5   |1      |
+-----------------------------+---------+--------+------+-------+
only showing top 10 rows

## Extract User Details

In [8]:
user_id = 128058

user_details = data.filter(col("UserID") == user_id).select("City", "Category").distinct().collect()[0]
user_city = user_details["City"]
user_category = user_details["Category"]

print(f"User City: {user_city}, User Category: {user_category}")


VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

User City: Chicago, User Category: Delivery

## Find User's Cluster

In [9]:
user_cluster = clustered_data.filter(
    (col("UserID") == user_id) & 
    (col("City") == user_city) & 
    (col("Category") == user_category)
).select("cluster").first()["cluster"]

print(f"User's Cluster: {user_cluster}")


VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

User's Cluster: 2

## Recommend Items for the User

In [10]:
recommendations = clustered_data.filter(
    (col("cluster") == user_cluster) & 
    (col("City") == user_city) & 
    (col("Category") == user_category)
).select(
    "ItemID", "Rating", "NumberReview", "City", "Category"
).dropDuplicates(["ItemID"]).orderBy(col("Rating").desc())

recommendations.show(10, truncate=False)


VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------------------------+------+------------+-------+--------+
|ItemID                    |Rating|NumberReview|City   |Category|
+--------------------------+------+------------+-------+--------+
|Crisp                     |4.5   |3228        |Chicago|Delivery|
|Eataly Chicago            |4.0   |3749        |Chicago|Delivery|
|Giordano's                |4.0   |2790        |Chicago|Delivery|
|Piece Brewery and Pizzeria|4.0   |3560        |Chicago|Delivery|
|Sunda Chicago             |4.0   |2835        |Chicago|Delivery|
|Xoco                      |4.0   |3646        |Chicago|Delivery|
+--------------------------+------+------------+-------+--------+

## Evaluate Clustering

In [11]:
cluster_sizes = clustered_data.groupBy("cluster").count().orderBy("count", ascending=False)
cluster_sizes.show()


VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-------+------+
|cluster| count|
+-------+------+
|      1|734048|
|      7|128120|
|      0| 63373|
|      9| 35996|
|      8| 20372|
|      5| 10043|
|      4|  4588|
|      3|  1798|
|      2|   487|
|      6|   117|
+-------+------+

In [26]:
cluster_sizes.write.csv("s3://yelp-final-raja/cluster_sizes.csv", mode="overwrite", header=True)


VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…